In [1]:
from data_model_loader import *

model = load_model()
images = load_coco_2014_dataset()

# load file paths
with open('config.json', 'r') as file:
    config = json.load(file)
coco_folder = config['coco_folder']
annotation_file_path = config['annotation_file_path']

# filename_to_image_id = get_image_ids()


OD model already present locally.
data/coco2014\val2014.zip already exists. Skipping download.
data/coco2014\val2014 already exists. Skipping extraction.
data/coco2014\annotations_trainval2014.zip already exists. Skipping download.
data/coco2014\annotations_trainval2014 already exists. Skipping extraction.
COCO 2014 dataset download and extraction complete.
Randomly selecting 1500 pictures ..
Done!


Original Model

In [2]:
from model import *

# load image
image_path = r"data\coco2014\val2014\val2014\COCO_val2014_000000322029.jpg"
image = Image.open(image_path)

image_tensor = preprocess_image_saved_model(image)

# inference
detector_output = model(image_tensor)

detector_output

{'detection_multiclass_scores': <tf.Tensor: shape=(1, 100, 91), dtype=float32, numpy=
 array([[[0.00472191, 0.8726634 , 0.00392656, ..., 0.02455736,
          0.0099301 , 0.00899006],
         [0.00610716, 0.1252546 , 0.06074021, ..., 0.02028845,
          0.04464232, 0.0284424 ],
         [0.00497259, 0.8172499 , 0.01470115, ..., 0.02694433,
          0.011177  , 0.01327647],
         ...,
         [0.00414066, 0.12198101, 0.02566327, ..., 0.02857963,
          0.01321094, 0.07116009],
         [0.00465236, 0.5033082 , 0.08573559, ..., 0.04113628,
          0.01719933, 0.04576527],
         [0.00358413, 0.09629517, 0.03092998, ..., 0.00650542,
          0.00342711, 0.0021734 ]]], dtype=float32)>,
 'detection_classes': <tf.Tensor: shape=(1, 100), dtype=float32, numpy=
 array([[ 1., 19.,  1.,  2.,  1.,  1.,  3.,  1.,  1.,  1.,  1.,  1.,  1.,
          1.,  1.,  1.,  3.,  1.,  1.,  3.,  1.,  1.,  8.,  3., 19.,  1.,
          1., 77.,  1.,  3., 77.,  1., 19.,  8.,  2.,  1.,  2.,  8.,  3.,

Tflite float32 Model

In [8]:
import tensorflow as tf
import pathlib
from tqdm import tqdm
import torchvision.transforms as transforms
from PIL import Image
import numpy as np

def load_tflite_model(tflite_model_path):
    # Load the TFLite model and allocate tensors.
    interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
    interpreter.allocate_tensors()
    
    # Get input and output tensors.
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    return interpreter, input_details, output_details

def preprocess_image(image_path, input_details):
    # Open image and convert to RGB
    image = Image.open(image_path).convert('RGB')
    width, height = image.size
    
    # Resize the image to the expected size
    target_shape = input_details[0]['shape'][1:3]  # height, width
    image = image.resize(target_shape)
    
    # Convert to numpy array and scale to [0, 255]
    image_np = np.array(image, dtype=np.uint8)
    
    # Add batch dimension [1, height, width, 3]
    image_np = np.expand_dims(image_np, axis=0)
    
    return image_np, width, height

def run_inference(interpreter, input_details, output_details, image, height, width):
    # Set the input tensor
    interpreter.set_tensor(input_details[0]['index'], image)
    
    # Run inference
    interpreter.invoke()
    
    # Get the output results
    output_data = {}
    for output_detail in output_details:
        output_data[output_detail['name']] = interpreter.get_tensor(output_detail['index'])
        output_data["image_size"] = [width, height]
    
    return output_data


def process_output(output_data):
    num_detections = output_data['StatefulPartitionedCall:5']  # correct
    detection_boxes = output_data['StatefulPartitionedCall:1'][0] # probably correct
    detection_classes = output_data['StatefulPartitionedCall:2'][0]  # correct
    detection_scores = output_data['StatefulPartitionedCall:4'][0]   # correct
    
    # Convert class index and score to a list of detections
    num_detections = int(num_detections[0])

    print(detection_scores[0])

    results = []
    for i in range(num_detections):
        ymin, xmin, ymax, xmax = detection_boxes[i]
        ymin = ymin * height
        ymax = ymax * height
        xmin = xmin * width
        xmax = xmax * width

        result = {
                    "image_id" : "None",
                    "category_id": int(detection_classes[i]),
                    "bbox": [xmin, ymin, xmax - xmin, ymax - ymin], # detector_output["detection_boxes"].numpy()[0][i].tolist(), # needs to be list
                    "score": float(detection_scores[i])
        }
    
        results.append(result)

    
    return results



tflite_model_path = "model/ssd_mobilenet_quantized/ssd_mobilenet_float32.tflite"
image_path = r"data\coco2014\val2014\val2014\COCO_val2014_000000322029.jpg"

interpreter, input_details, output_details = load_tflite_model(tflite_model_path)
image_np, width, height = preprocess_image(image_path, input_details)


# !!! Dict uses generic keys(), they are different to from orignal/saved_model 
output_data = run_inference(interpreter, input_details, output_details, image_np,width, height)
output_data
# !!! This creates desired result json
#detections = process_output(output_data)
#detections

{'StatefulPartitionedCall:6': array([[[-9.8323692e-03,  5.6441687e-04,  4.1892581e-02,  3.5092913e-02],
         [-1.0471346e-02, -8.0547720e-02,  6.8331122e-02,  1.6212142e-01],
         [-6.2480062e-02, -2.7799180e-02,  2.1418935e-01,  8.8875301e-02],
         ...,
         [ 1.9614771e-01, -1.3405621e-02,  8.2933044e-01,  1.0137303e+00],
         [-5.8508515e-03,  2.0341018e-01,  1.0102913e+00,  8.1383216e-01],
         [ 5.5006862e-02,  4.1899443e-02,  9.9047005e-01,  9.7592103e-01]]],
       dtype=float32),
 'image_size': [627, 640],
 'StatefulPartitionedCall:0': array([[1916., 1911., 1916., 1784., 1916., 1916., 1754., 1916., 1907.,
         1724., 1916., 1730., 1911., 1818., 1916., 1916., 1878., 1902.,
         1784., 1226., 1916., 1862., 1906., 1724., 1911., 1250., 1784.,
         1862., 1784., 1916., 1779., 1876., 1912., 1790., 1784., 1916.,
         1893., 1784., 1644., 1796., 1862., 1862., 1520., 1916., 1911.,
         1916., 1730., 1878., 1796., 1550., 1730., 1861., 1784., 1

In [9]:
interpreter.get_output_details()

[{'name': 'StatefulPartitionedCall:6',
  'index': 398,
  'shape': array([   1, 1917,    4]),
  'shape_signature': array([   1, 1917,    4]),
  'dtype': numpy.float32,
  'quantization': (0.0, 0),
  'quantization_parameters': {'scales': array([], dtype=float32),
   'zero_points': array([], dtype=int32),
   'quantized_dimension': 0},
  'sparsity_parameters': {}},
 {'name': 'StatefulPartitionedCall:0',
  'index': 2145,
  'shape': array([  1, 100]),
  'shape_signature': array([ 1, -1]),
  'dtype': numpy.float32,
  'quantization': (0.0, 0),
  'quantization_parameters': {'scales': array([], dtype=float32),
   'zero_points': array([], dtype=int32),
   'quantized_dimension': 0},
  'sparsity_parameters': {}},
 {'name': 'StatefulPartitionedCall:5',
  'index': 2110,
  'shape': array([1]),
  'shape_signature': array([1]),
  'dtype': numpy.float32,
  'quantization': (0.0, 0),
  'quantization_parameters': {'scales': array([], dtype=float32),
   'zero_points': array([], dtype=int32),
   'quantized_dim

Find correct keys()
- I compared the shapes and the values 
- https://www.kaggle.com/models/tensorflow/ssd-mobilenet-v2/tensorFlow2/ssd-mobilenet-v2/1?tfhub-redirect=true